# **Training Pipeline**

Quy trình huấn luyện seq2seq cho tóm tắt văn bản tiếng Việt

### Lưu ý quan trọng khi dùng TPU:
- Dùng **bf16** (TPU v5e hỗ trợ native bfloat16)
- **Không dùng** CUDA quantization (BitsAndBytes không hỗ trợ TPU)
- Cần cài **torch_xla** để PyTorch giao tiếp với TPU
- HuggingFace Trainer tự động detect TPU khi có torch_xla

In [9]:
from __future__ import annotations
import os

# Xác định TPU bằng torch_xla; TPU Kaggle có thể không đặt TPU_NAME.
tpu_name = os.environ.get("TPU_NAME", os.environ.get("COLAB_TPU_ADDR"))
is_tpu = False
print(f"TPU_NAME: {tpu_name}")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm

    device = xm.xla_device()
    is_tpu = str(device).startswith("xla")
    print(f"✅ TPU sẵn sàng! Device: {device}")
    print(f"   torch_xla version: {torch_xla.__version__}")
except Exception as e:
    print(f"ℹ️ Không dùng TPU: {e}")

print(f"Runtime đang dùng: {'TPU' if is_tpu else 'GPU/CPU'}")

TPU_NAME: None
✅ TPU sẵn sàng! Device: xla:0
   torch_xla version: 2.8.0


In [10]:
from __future__ import annotations
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"

# Khắc phục lỗi clone lồng nhau (lỗi 2 thư mục sumarization)
if os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code mới nhất...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

Đang cập nhật code mới nhất...
Already up to date.


In [11]:
%cd /kaggle/working/sumarization

import os
import sys
from importlib import invalidate_caches
from importlib.metadata import version

# Nếu cell này chạy sau lỗi tokenizer, Transformers v5 cũ vẫn còn trong RAM.
transformers_was_loaded = "transformers" in sys.modules

# Pipeline dùng PyTorch/XLA; tắt TensorFlow/Flax để tránh lỗi safetensors của image Kaggle.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

# Dùng mã nguồn hiện tại nhưng không để pip thay đổi dependency của runtime.
!{sys.executable} -m pip install -q --no-deps -e .

# Chỉ TPU cần bộ tokenizer cũ tương thích với ViT5; GPU giữ nguyên môi trường đang chạy tốt.
if is_tpu:
    !{sys.executable} -m pip install -q --no-cache-dir --force-reinstall --no-deps "transformers==4.51.3" "tokenizers==0.21.4" "huggingface-hub==0.30.2" "sentencepiece==0.2.0"

    print(f"Transformers đã cài: {version('transformers')}")
    print(f"Tokenizers đã cài:   {version('tokenizers')}")

    if transformers_was_loaded:
        # Bỏ module cũ trong RAM để import lại từ phiên bản vừa cài, không restart TPU session.
        module_prefixes = ("transformers", "tokenizers", "huggingface_hub")
        for module_name in list(sys.modules):
            if module_name in module_prefixes or module_name.startswith(
                tuple(f"{prefix}." for prefix in module_prefixes)
            ):
                sys.modules.pop(module_name, None)

        # Các module này giữ reference đến Transformers cũ, nên cần import lại ở cell kế tiếp.
        for module_name in ("src.model", "src.predict", "src.trainer"):
            sys.modules.pop(module_name, None)

        invalidate_caches()
        import transformers
        import tokenizers

        print(f"🔄 Đã nạp lại: transformers={transformers.__version__}, tokenizers={tokenizers.__version__}")
        print("➡️ Không cần restart TPU. Chạy lại hai cell import rồi tiếp tục từ cell cấu hình.")
    else:
        print("✅ TPU đã sẵn sàng dùng tokenizer ViT5 tương thích.")
else:
    print("✅ GPU/CPU: giữ nguyên phiên bản Transformers và Tokenizers hiện có.")

/kaggle/working/sumarization

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vn-summarization 1.0.0 requires evaluate>=0.4.2, which is not installed.
vn-summarization 1.0.0 requires peft>=0.12.0, which is not installed.
vn-summarization 1.0.0 requires rouge-score>=0.1.2, which is not installed.
vn-summarization 1.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.21.4 which is incompatible.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.
google-tunix 0.1.7 requires safetensors<0.8, but you have safetensors 0.8.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [12]:
import warnings
warnings.filterwarnings('ignore')  # Ẩn các cảnh báo phiền phức

import time
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [13]:
from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

✅ Import thành công!


In [14]:
CONFIG_FILE = "configs/vit5_warmstart_phase_1.yaml"
config = load_config(CONFIG_FILE)

colab_output_model = Path("/content") / config.training.output_dir
kaggle_output_model = Path("/kaggle/working") / config.training.output_dir

is_kaggle = Path("/kaggle").exists()

output_model = (
    kaggle_output_model
    if is_kaggle
    else colab_output_model
)

best_dir = output_model / "best"

# Cập nhật lại config để Trainer lưu đúng đường dẫn tuyệt đối
config = apply_overrides(config, {
    "training.output_dir": str(output_model),
})

print(f"Colab output model: {colab_output_model}")
print(f"Kaggle output model: {kaggle_output_model}")
print(f"Output đang sử dụng: {output_model}")
print(f"Best checkpoint: {best_dir}")
print(config.data)

Colab output model: /content/outputs_phase_1/vit5_warmstart
Kaggle output model: /kaggle/working/outputs_phase_1/vit5_warmstart
Output đang sử dụng: /kaggle/working/outputs_phase_1/vit5_warmstart
Best checkpoint: /kaggle/working/outputs_phase_1/vit5_warmstart/best
DataConfig(train_file='data/phase_1/train_*.**', valid_file='data/phase_1/validation_*.**', test_file='data/phase_1/test_*.**', source_prefix='summarize: ', max_source_length=768, max_target_length=256, max_train_samples=None, max_eval_samples=None)


---
## 3. Cấu hình phần cứng  (đọc phần cứng để áp dụng gput hoặc tpu)

In [15]:
# Chọn config cơ sở


# Tự động phát hiện đang dùng TPU hay GPU
is_tpu = bool(globals().get("is_tpu", False)) or bool(
    os.environ.get("TPU_NAME") or os.environ.get("COLAB_TPU_ADDR")
)

# Ghi đè các thiết lập cho Lần 1
config = apply_overrides(config, {

    # Precision & Optimizer
    "training.precision": "bf16" if is_tpu else "auto",
    "training.optim": "adafactor" if is_tpu else "adamw_torch",

    # Thư mục lưu kết quả Lần 1
    "training.output_dir": output_model,

    # Batch size
    "training.per_device_train_batch_size": 8 if is_tpu else 4,
    "training.per_device_eval_batch_size": 16 if is_tpu else 8,
    "training.gradient_accumulation_steps": 1 if is_tpu else 2,

    # --- CỤM CHẠY THỬ ---
    # "data.max_train_samples": 200,   
    # "data.max_eval_samples": 50,     
    # "training.max_steps": 20,
    # "training.logging_steps": 2,     # <--- Thêm ở đây
    # "training.eval_steps": 5,        # <--- Thêm ở đây

})

print(f"🔹 TRAIN LẦN {config.phase.name}: Học nền tảng (Parquet tổng hợp)")
print(f"Hardware:   {'TPU' if is_tpu else 'GPU'}")
print(f"Model:      {config.model.name_or_path}")
print(f"Train data: {config.data.train_file}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"LR:         {config.training.learning_rate}")
print(f"Optimizer:  {config.training.optim}")
print(f"Output:     {config.training.output_dir}")


🔹 TRAIN LẦN phase_1: Học nền tảng (Parquet tổng hợp)
Hardware:   GPU
Model:      VietAI/vit5-base-vietnews-summarization
Train data: data/phase_1/train_*.**
Precision:  auto
Batch size: 4
LR:         1e-05
Optimizer:  adamw_torch
Output:     /kaggle/working/outputs_phase_1/vit5_warmstart


---
## 4. Tải Model & Data

In [16]:
# Thiết lập seed
set_seed(config.training.seed)

# Tải tokenizer
tokenizer = load_tokenizer(config.model)
print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

[INFO] src.model: Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base-vietnews-summarization
INFO:src.model:Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base-vietnews-summarization
[WARNING] src.model: T5Tokenizer thất bại (argument 'vocab': 'dict' object is not an instance of 'Sequence'), chuyển về AutoTokenizer


TypeError: argument 'vocab': 'dict' object is not an instance of 'Sequence'

In [ ]:
# Tải model
model = load_model(config.model, tokenizer, config.generation)

# Áp dụng gradient checkpointing (tiết kiệm bộ nhớ TPU)
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

# Đóng băng encoder nếu cần
if config.training.freeze_encoder:
    freeze_encoder(model)

# Áp dụng LoRA nếu bật
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(f"✅ Model loaded: {format_number(params['total'])} total, "
      f"{format_number(params['trainable'])} trainable ({params['trainable_percent']}%)")

In [ ]:
# Tải & tiền xử lý dữ liệu
datasets = load_and_preprocess(tokenizer, config.data)

print(f"✅ Train:      {len(datasets['train'])} samples")
print(f"✅ Validation: {len(datasets['validation'])} samples")
if 'test' in datasets:
    print(f"✅ Test:       {len(datasets['test'])} samples")


---
## 5. Cấu hình Trainer

In [ ]:
tc = config.training
output_dir = Path(tc.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Xác định precision
fp16 = tc.precision == "fp16"
bf16 = tc.precision == "bf16"

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,

    # Epochs & steps
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,

    # Batch size
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,

    # Optimizer
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,  #thuật toán cập nhật kiến thức

    # Regularization
    label_smoothing_factor=tc.label_smoothing_factor,

    # Precision — TPU v5e dùng bf16
    fp16=fp16,
    bf16=bf16,

    # Evaluation & saving
    eval_strategy=tc.eval_strategy,  #số bước học cho mỗi lần kiểm tra
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,  # số bước học cho mỗi lần lưu
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,  # số bản lưu trữ được giữ lại
    logging_steps=tc.logging_steps,

    # Best model
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,  #chọn bản tốt nhất để lưu

    # Generation during eval
    predict_with_generate=True,  #yêu cầu tạo ra 1 bài báo và tóm tắt rồi mới chấm điểm
    generation_max_length=config.generation.max_length,  #số token tối đa được viết khi làm bài kiểm tra

    # Logging — dùng tensorboard
    report_to=["tensorboard"],
    logging_dir=str(output_dir / "logs"),

    # ===== TPU-specific =====
    # dataloader_drop_last giúp tránh batch cuối bị lẻ trên TPU
    dataloader_drop_last=True,
)

print(f"✅ TrainingArguments đã sẵn sàng")
print(f"   bf16={bf16}, fp16={fp16}")
print(f"   device: {training_args.device}")

In [ ]:
# Data collator (dynamic padding)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# Hàm tính ROUGE metrics
compute_metrics = build_compute_metrics(tokenizer)

# Callbacks
callbacks = []
if tc.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(
            early_stopping_patience=tc.early_stopping_patience,
        )
    )
    print(f"✅ Early stopping: patience={tc.early_stopping_patience}")

# Xây dựng Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("✅ Trainer đã sẵn sàng!")

---
## 6. Huấn luyện 🚀

In [ ]:
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 60)
print(f"  Model:       {config.model.name_or_path}")
print(f"  Epochs:      {tc.num_train_epochs}")
print(f"  Batch size:  {tc.per_device_train_batch_size}")
print(f"  Grad accum:  {tc.gradient_accumulation_steps}")
print(f"  LR:          {tc.learning_rate}")
print(f"  Optimizer:   {tc.optim}")
print(f"  Precision:   {tc.precision}")
print(f"  LoRA:        {config.lora.enabled}")
print("=" * 60)

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=tc.resume_from_checkpoint,
)

elapsed = time.time() - start_time
print(f"\n✅ Huấn luyện hoàn thành! Thời gian: {format_duration(elapsed)}")

---
## 7. Lưu model & Đánh giá

In [ ]:
# Lưu model tốt nhất

print(f"Đang lưu model tốt nhất tới: {best_dir}")
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print("✅ Đã lưu model!")

In [ ]:
# Đánh giá cuối cùng trên tập VALIDATION
print("Đang chạy đánh giá trên tập Validation...")
eval_results = trainer.evaluate(metric_key_prefix="eval")

print("\n" + "=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ - TẬP VALIDATION")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print("=" * 60)

# Đánh giá trên tập TEST (Bài thi Đại học - Độc lập hoàn toàn)
if 'test' in datasets:
    print("\nĐang đánh giá trên tập TEST...")
    test_results = trainer.evaluate(eval_dataset=datasets['test'], metric_key_prefix='test')
    print("\n" + "=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ - TẬP TEST (Khách quan nhất)")
    print("=" * 60)
    for k, v in sorted(test_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
    print("=" * 60)
else:
    print("\n⚠️ Không tìm thấy tập TEST trong dữ liệu.")
    test_results = {}


In [ ]:
# Lưu metrics & config
train_metrics = train_result.metrics
train_metrics["train_runtime_formatted"] = format_duration(
    train_metrics.get("train_runtime", 0)
)

save_json(train_metrics, output_dir / "train_results.json")
save_json(eval_results, output_dir / "eval_results.json")
if test_results:
    save_json(test_results, output_dir / "test_results.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")

print("✅ Đã lưu tất cả metrics!")
print(f"   📁 {output_dir}")


---
## 8 Test thử model

In [ ]:
from src.predict import summarize

test_text = """Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng 
trí tuệ nhân tạo tại Việt Nam giai đoạn 2025-2030. Theo đó, Việt Nam đặt 
mục tiêu trở thành một trong những trung tâm đổi mới sáng tạo về AI trong 
khu vực ASEAN. Đề án tập trung vào 5 lĩnh vực ưu tiên gồm y tế, giáo dục, 
nông nghiệp, giao thông và sản xuất công nghiệp."""

summary = summarize(
    text=test_text,
    model_path=str(best_dir),
    config=config,
)

print("📄 Bài gốc:")
print(test_text.strip())
print("\n📝 Tóm tắt:")
print(summary)